# Synthetic Chinese hamster ovary fed-batch culture time-series profiles, process conditions, and benchmark results Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In this notebook, we will:
- load the dataset metadata from its Croissant schema,
- inspect available record sets and fields using their `@id` values,
- load selected record sets into pandas DataFrames,
- perform exploratory data analysis on numeric and categorical fields,
- create example visualizations for time-series and summary distributions.

> Important: all references to record sets and fields in this notebook are handled through their Croissant `@id` values whenever available.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import json
from pprint import pprint

import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 200)
sns.set_theme(style='whitegrid')

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vfyp-4wea/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")
print(f"Citation: {getattr(metadata, 'cite_as', getattr(metadata, 'citeAs', 'N/A'))}")

In [ ]:
# Convert metadata to JSON for flexible inspection, without subscripting the metadata object itself.
metadata_json = metadata.to_json()

summary_keys = [
    'name', 'description', 'identifier', 'version', 'datePublished',
    'license', 'keywords', 'temporalCoverage', 'dataUseCases'
]

metadata_summary = {k: metadata_json.get(k) for k in summary_keys if k in metadata_json}
pprint(metadata_summary)

## 2. Data Overview
Review available record sets, fields, and their IDs.

Because Croissant datasets can expose multiple linked tables, we first inspect the dataset structure programmatically. This is especially useful when building reusable data-loading pipelines.

In [ ]:
# Inspect top-level metadata JSON keys
print('Top-level metadata keys:')
print(sorted(metadata_json.keys()))

record_sets_json = metadata_json.get('recordSet', [])
print(f"\nNumber of record sets declared in metadata JSON: {len(record_sets_json)}")

In [ ]:
# Helper function to inspect record set metadata using @id fields.
def extract_recordset_overview(metadata_json_obj):
    rows = []
    for rs in metadata_json_obj.get('recordSet', []):
        rs_id = rs.get('@id', 'N/A')
        rs_name = rs.get('name', '')
        rs_desc = rs.get('description', '')
        fields = rs.get('field', [])
        rows.append({
            'record_set_id': rs_id,
            'record_set_name': rs_name,
            'field_count': len(fields),
            'description': rs_desc
        })
    return pd.DataFrame(rows)

recordset_overview_df = extract_recordset_overview(metadata_json)
recordset_overview_df

In [ ]:
# If record sets are available, inspect their fields by @id.
field_rows = []
for rs in metadata_json.get('recordSet', []):
    rs_id = rs.get('@id', 'N/A')
    rs_name = rs.get('name', '')
    for field in rs.get('field', []):
        field_rows.append({
            'record_set_id': rs_id,
            'record_set_name': rs_name,
            'field_id': field.get('@id', 'N/A'),
            'field_name': field.get('name', ''),
            'data_type': field.get('dataType', ''),
            'source': field.get('source', '')
        })

fields_overview_df = pd.DataFrame(field_rows)
print(f"Number of fields discovered: {len(fields_overview_df)}")
fields_overview_df.head(20)

If the dataset exposes record sets, the next cell demonstrates how to preview records from a specific record set using its `@id`. The code is written defensively so the notebook remains usable even when the record set list is empty or inaccessible from the metadata.

In [ ]:
# Example preview of records from the first available record set, if any.
available_record_set_ids = recordset_overview_df['record_set_id'].tolist() if not recordset_overview_df.empty else []

if available_record_set_ids:
    example_record_set_id = available_record_set_ids[0]
    print(f"Previewing records from record set @id: {example_record_set_id}")
    for i, x in enumerate(dataset.records(record_set=example_record_set_id)):
        print(x)
        if i >= 2:
            break
else:
    print('No record sets were declared in metadata_json["recordSet"].')
    print('This dataset may expose metadata only at the current URL, or record sets may be resolved differently by the provider.')

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview.

The extraction logic below uses variables so you can easily switch record sets without rewriting the workflow.

In [ ]:
# Extract data from each discovered record set
record_sets = available_record_set_ids
dataframes = {}
record_counts = []

for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        record_counts.append({
            'record_set_id': record_set_id,
            'n_rows': len(df),
            'n_columns': len(df.columns)
        })
    except Exception as e:
        print(f"Could not load record set {record_set_id}: {e}")

record_counts_df = pd.DataFrame(record_counts)
record_counts_df

In [ ]:
# Inspect columns and a preview for one loaded record set.
if dataframes:
    selected_record_set_id = next(iter(dataframes))
    print(f"Selected record set @id: {selected_record_set_id}")
    print('Columns:')
    print(dataframes[selected_record_set_id].columns.tolist())
    display(dataframes[selected_record_set_id].head())
else:
    print('No record sets were loaded into DataFrames.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

This section is designed to work dynamically:
- it identifies a loaded record set,
- finds numeric and categorical columns,
- filters records on a numeric threshold,
- normalizes the selected numeric field,
- groups data by a suitable categorical field when possible.

In [ ]:
def choose_analysis_columns(df):
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    object_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
    return numeric_cols, object_cols

if dataframes:
    record_set_id = next(iter(dataframes))
    analysis_df = dataframes[record_set_id].copy()
    
    # Attempt to coerce potentially numeric object columns.
    for col in analysis_df.columns:
        if analysis_df[col].dtype == 'object':
            analysis_df[col] = pd.to_numeric(analysis_df[col], errors='ignore')
    
    numeric_columns, group_columns = choose_analysis_columns(analysis_df)
    print(f"Working record set @id: {record_set_id}")
    print(f"Numeric columns found: {numeric_columns[:10]}")
    print(f"Candidate grouping columns: {group_columns[:10]}")
else:
    analysis_df = pd.DataFrame()
    record_set_id = None
    numeric_columns, group_columns = [], []
    print('No data available for EDA.')

In [ ]:
# Select a numeric field for analysis dynamically
if not analysis_df.empty and numeric_columns:
    numeric_field = numeric_columns[0]
    threshold = analysis_df[numeric_field].median()
    
    filtered_df = analysis_df[analysis_df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())
    
    std_val = filtered_df[numeric_field].std()
    if pd.notna(std_val) and std_val != 0:
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / std_val
        )
    else:
        filtered_df[f"{numeric_field}_normalized"] = 0.0
    
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    
    group_field = group_columns[0] if group_columns else None
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field}:")
        display(grouped_df.head())
    else:
        grouped_df = pd.DataFrame()
        print('No suitable grouping field was found.')
else:
    filtered_df = pd.DataFrame()
    grouped_df = pd.DataFrame()
    numeric_field = None
    group_field = None
    print('No numeric columns were found for analysis.')

In [ ]:
# Additional overview of missing values and basic numeric statistics
if not analysis_df.empty:
    print('Missing values by column:')
    display(analysis_df.isna().sum().sort_values(ascending=False).head(20))
    
    numeric_summary = analysis_df.describe(include=[np.number]).T if numeric_columns else pd.DataFrame()
    print('Numeric summary statistics:')
    display(numeric_summary.head(20))
else:
    print('No analysis DataFrame available.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

The following examples generate visualizations only when compatible columns are available.

In [ ]:
if not analysis_df.empty and numeric_field is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(analysis_df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print('Histogram skipped because no numeric field is available.')

In [ ]:
if not filtered_df.empty and group_field is not None and not grouped_df.empty:
    plt.figure(figsize=(10, 4))
    sns.barplot(data=grouped_df.head(20), x=group_field, y=numeric_field)
    plt.title(f'Average {numeric_field} by {group_field}')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print('Grouped bar plot skipped because grouped data is unavailable.')

In [ ]:
# Scatter plot for the first two numeric columns, if available
if not analysis_df.empty and len(numeric_columns) >= 2:
    x_field = numeric_columns[0]
    y_field = numeric_columns[1]
    
    plt.figure(figsize=(6, 5))
    sns.scatterplot(data=analysis_df, x=x_field, y=y_field, alpha=0.7)
    plt.title(f'{y_field} vs {x_field}')
    plt.tight_layout()
    plt.show()
else:
    print('Scatter plot skipped because fewer than two numeric columns are available.')

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

This notebook demonstrated a reusable workflow for exploring a Croissant-described dataset with `mlcroissant`.

Key steps included:
- loading dataset metadata from the Croissant URL,
- inspecting record sets and fields using their `@id` values,
- extracting record sets into pandas DataFrames,
- performing basic filtering, normalization, grouping, and descriptive analysis,
- generating visualizations for numeric distributions and relationships.

For this dataset in particular, the metadata describes a synthetic CHO fed-batch culture benchmarking resource with hierarchical scenario, culture, and measurement structure. When record sets are available through the Croissant interface, the same notebook can be extended to:
- compare latent versus observed trajectories,
- examine censoring indicators,
- analyze process-condition effects,
- evaluate benchmark result tables.

A recommended next step is to identify the specific record set `@id`s corresponding to process scenarios, latent trajectories, observed measurements, and benchmark outputs, then repeat the extraction and analysis sections for those linked tables.